# 🧪 DQ Framework — Notebook 5: Tests

**Purpose:** Self-contained test suite that runs directly in Databricks.
No pytest required — tests use plain Python assertions and `spark.createDataFrame`.

**How to run:** Open this notebook in Databricks and click **Run All**.
All cells must complete without a red error banner.

**What is tested:**
* Config validation (all 16 rule types, error cases, duplicate detection)
* Rule engine (all 16 rule types against synthetic Spark DataFrames)
* Dataset loader (source type routing)

## 0. Setup — load helper notebooks and install deps

In [0]:
%pip install pyyaml --quiet

In [0]:
%run ./01_dq_config_validator

In [0]:
%run ./02_dq_rule_engine

In [0]:
%run ./03_dq_dataset_loader

In [0]:
import tempfile, yaml, os
from pathlib import Path
from datetime import datetime, timedelta, timezone

# Test counters
_passed = 0
_failed = 0

def _ok(label):
    global _passed
    _passed += 1
    print(f"  ✅ PASS  {label}")

def _fail(label, detail=""):
    global _failed
    _failed += 1
    detail_str = f"\n         → {detail}" if detail else ""
    print(f"  ❌ FAIL  {label}{detail_str}")

def _section(title):
    print(f"\n{'─' * 60}")
    print(f"  {title}")
    print(f"{'─' * 60}")

## 1. Config Validation Tests

In [0]:
_section("CONFIG VALIDATION")

def _mk_ds_cfg(name):
    return {
        "dataset": {
            "name": name,
            "description": f"Test dataset {name}",
            "source": {"type": "delta", "path": f"main.test.{name}"},
        },
        "rules": [{
            "id": f"{name}_pk",
            "description": "PK check",
            "type": "not_null",
            "column": "id",
            "severity": "CRITICAL",
        }],
    }

def _vr(rule):
    return _validate_rule(rule, 0, "test.yml", set())

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)

    def _wy(rel, content):
        p = tmp / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(yaml.dump(content))
        return str(p)

    def _mp(entries, gs=None):
        return _wy("config/mc.yml", {"datasets": entries, "global_settings": gs or {}})

    def _de(name, file, enabled=True):
        return {"name": name, "config_file": file, "enabled": enabled}

    # 1.1 Valid master config
    try:
        _wy("config/datasets/o.yml", _mk_ds_cfg("orders"))
        res = load_master_config(_mp([_de("orders", "config/datasets/o.yml")]))
        assert len(res["datasets"]) == 1
        _ok("Valid master config loads successfully")
    except Exception as e:
        _fail("Valid master config", e)

    # 1.2 Duplicate names → DQConfigError
    try:
        try:
            load_master_config(_mp([_de("orders", "a.yml"), _de("orders", "b.yml")]))
            _fail("Duplicate names — no exception raised")
        except DQConfigError as e:
            assert "DUPLICATE" in str(e)
            _ok("Duplicate dataset names raise DQConfigError")
    except Exception as e:
        _fail("Duplicate names test", e)

    # 1.3 Multiple duplicates all reported in one error
    try:
        try:
            load_master_config(_mp([
                _de("orders", "a.yml"), _de("orders", "b.yml"),
                _de("customers", "c.yml"), _de("customers", "d.yml"),
            ]))
            _fail("Multiple duplicates — no exception raised")
        except DQConfigError as e:
            err = str(e)
            assert "'orders'" in err and "'customers'" in err
            _ok("Multiple duplicate names all reported in one error")
    except Exception as e:
        _fail("Multiple duplicates test", e)

    # 1.4 Missing config_file field
    try:
        try:
            load_master_config(_wy("config/m2.yml", {
                "datasets": [{"name": "orders", "enabled": True}],
                "global_settings": {},
            }))
            _fail("Missing config_file — no exception raised")
        except DQConfigError as e:
            assert "config_file" in str(e)
            _ok("Missing config_file field raises DQConfigError")
    except Exception as e:
        _fail("Missing config_file test", e)

    # 1.5 Config file not found on disk
    try:
        try:
            load_dataset_config(_de("orders", "config/datasets/NONEXISTENT.yml"), str(tmp))
            _fail("Missing file — no exception raised")
        except DQDatasetConfigError as e:
            assert "not found on disk" in str(e)
            _ok("Missing config_file on disk raises DQDatasetConfigError")
    except Exception as e:
        _fail("Missing file test", e)

    # 1.6 Name mismatch shows both names
    try:
        _wy("config/datasets/wrong.yml", _mk_ds_cfg("WRONG_NAME"))
        try:
            load_dataset_config(_de("orders", "config/datasets/wrong.yml"), str(tmp))
            _fail("Name mismatch — no exception raised")
        except DQDatasetConfigError as e:
            err = str(e)
            assert "NAME MISMATCH" in err and "orders" in err and "WRONG_NAME" in err
            _ok("Name mismatch shows both names in error message")
    except Exception as e:
        _fail("Name mismatch test", e)

    # 1.7 Empty rules block
    try:
        bad = _mk_ds_cfg("orders")
        bad["rules"] = []
        _wy("config/datasets/empty_rules.yml", bad)
        try:
            load_dataset_config(_de("orders", "config/datasets/empty_rules.yml"), str(tmp))
            _fail("Empty rules — no exception raised")
        except DQDatasetConfigError as e:
            assert "rule" in str(e).lower()
            _ok("Empty rules block raises DQDatasetConfigError")
    except Exception as e:
        _fail("Empty rules test", e)

    # 1.8 All dataset errors collected before raising
    try:
        for n in ("ds_a", "ds_b", "ds_c"):
            _wy(f"config/datasets/{n}.yml", _mk_ds_cfg("MISMATCH"))
        try:
            load_all_configs(
                _mp([_de("ds_a", "config/datasets/ds_a.yml"),
                     _de("ds_b", "config/datasets/ds_b.yml"),
                     _de("ds_c", "config/datasets/ds_c.yml")]),
                str(tmp),
            )
            _fail("All errors — no exception raised")
        except DQConfigError as e:
            err = str(e)
            assert "ds_a" in err and "ds_b" in err and "ds_c" in err
            _ok("All dataset errors collected and reported together")
    except Exception as e:
        _fail("All errors collection test", e)

    # 1.9 Disabled datasets excluded
    try:
        _wy("config/datasets/o2.yml", _mk_ds_cfg("orders"))
        _wy("config/datasets/c2.yml", _mk_ds_cfg("customers"))
        cfgs, _ = load_all_configs(
            _mp([_de("orders", "config/datasets/o2.yml", True),
                 _de("customers", "config/datasets/c2.yml", False)]),
            str(tmp),
        )
        names = [c["dataset"]["name"] for c in cfgs]
        assert "orders" in names and "customers" not in names
        _ok("Disabled datasets are excluded from run")
    except Exception as e:
        _fail("Disabled datasets test", e)

    # 1.10 No enabled datasets → error
    try:
        _wy("config/datasets/o3.yml", _mk_ds_cfg("orders"))
        try:
            load_all_configs(
                _mp([_de("orders", "config/datasets/o3.yml", False)]),
                str(tmp),
            )
            _fail("No enabled datasets — no exception raised")
        except DQConfigError as e:
            assert "No enabled" in str(e)
            _ok("No enabled datasets raises DQConfigError")
    except Exception as e:
        _fail("No enabled datasets test", e)

In [0]:
# ── Rule-level config validation ─────────────────────────────────────────────
for rule_type, rule_extra in [
    ("not_null",    {"column": "id"}),
    ("unique",      {"columns": ["id"]}),
    ("row_count",   {"min": 1}),
    ("accepted_values", {"column": "status", "values": ["A", "B"]}),
    ("regex",       {"column": "email", "pattern": r".*@.*"}),
    ("range",       {"column": "amount", "min": 0}),
    ("date_range",  {"column": "dt"}),
    ("referential", {"column": "cid", "ref_table": "t", "ref_column": "id"}),
    ("freshness",   {"column": "ts", "max_hours": 24}),
    ("custom_sql",  {"sql": "SELECT true AS pass, 0 AS row_count FROM {table}"}),
    ("completeness", {"column": "x", "max_null_rate": 0.05}),
    ("duplicate_count", {"columns": ["id"]}),
    ("conditional_not_null", {"column": "disc", "condition_column": "type", "condition_value": "X"}),
    ("mutual_exclusivity",   {"columns": ["a", "b", "c"]}),
    ("character_set",        {"column": "ref", "charset_name": "alphanumeric"}),
    ("length_check",         {"column": "phone", "min_length": 7, "max_length": 15}),
]:
    try:
        rule = {"id": "r", "description": "x", "type": rule_type, "severity": "WARNING", **rule_extra}
        errors = _vr(rule)
        assert errors == [], f"Unexpected errors: {errors}"
        _ok(f"Valid '{rule_type}' rule passes config validation")
    except Exception as e:
        _fail(f"Valid '{rule_type}' rule", e)

# New rule type error cases
for label, rule, expected_substring in [
    ("conditional_not_null invalid op",      {"id":"r","description":"x","type":"conditional_not_null","column":"c","condition_column":"t","condition_value":"X","condition_operator":"EQUALS","severity":"CRITICAL"}, "condition_operator"),
    ("conditional_not_null in+scalar",        {"id":"r","description":"x","type":"conditional_not_null","column":"c","condition_column":"t","condition_value":"X","condition_operator":"in","severity":"CRITICAL"}, "condition_value"),
    ("mutual_exclusivity only 1 column",      {"id":"r","description":"x","type":"mutual_exclusivity","columns":["only"],"severity":"CRITICAL"}, "at least 2"),
    ("character_set missing both modes",      {"id":"r","description":"x","type":"character_set","column":"x","severity":"WARNING"}, "charset_name"),
    ("character_set both modes specified",    {"id":"r","description":"x","type":"character_set","column":"x","charset_name":"alphanumeric","allowed_chars":"ABC","severity":"WARNING"}, "not both"),
    ("character_set unknown charset_name",    {"id":"r","description":"x","type":"character_set","column":"x","charset_name":"UNKNOWN","severity":"WARNING"}, "Unknown"),
    ("length_check no min or max",            {"id":"r","description":"x","type":"length_check","column":"x","severity":"WARNING"}, "min_length"),
    ("length_check negative min",             {"id":"r","description":"x","type":"length_check","column":"x","min_length":-1,"severity":"WARNING"}, "non-negative"),
    ("length_check float min",                {"id":"r","description":"x","type":"length_check","column":"x","min_length":1.5,"severity":"WARNING"}, "non-negative"),
    ("invalid rule type",                     {"id":"r","description":"x","type":"UNKNOWN_TYPE","severity":"WARNING"}, "Invalid rule type"),
    ("invalid severity",                      {"id":"r","description":"x","type":"not_null","column":"id","severity":"BLOCKER"}, "Invalid severity"),
]:
    try:
        errors = _vr(rule)
        assert any(expected_substring in e for e in errors), f"Expected '{expected_substring}' in errors: {errors}"
        _ok(f"Config error detected: {label}")
    except Exception as e:
        _fail(f"Config error: {label}", e)

## 2. Rule Engine Tests (requires Spark)

In [0]:
_section("RULE ENGINE — Spark DataFrame Tests")

engine = RuleEngine(spark)

def _make_df(rows, schema=None):
    if schema:
        return spark.createDataFrame(rows, schema)
    return spark.createDataFrame(rows)

def _run(rule_dict, df, dataset_name="test_ds"):
    return engine.execute(rule_dict, df, dataset_name)

def _assert_pass(label, result):
    if result.passed and not result.error:
        _ok(label)
    else:
        _fail(label, f"status={result.status}, details={result.details}, error={result.error}")

def _assert_fail(label, result):
    if not result.passed and not result.error:
        _ok(label)
    else:
        _fail(label, f"status={result.status}, details={result.details}")

# ── not_null ──────────────────────────────────────────────────────────────
df_clean = spark.createDataFrame([(1,), (2,), (3,)], ["id"])
df_nulls = spark.createDataFrame([(1,), (None,), (3,)], ["id"])

_assert_pass("not_null: clean column passes",
    _run({"id":"r","description":"x","type":"not_null","column":"id","severity":"CRITICAL"}, df_clean))
_assert_fail("not_null: column with nulls fails",
    _run({"id":"r","description":"x","type":"not_null","column":"id","severity":"CRITICAL"}, df_nulls))

# ── unique ────────────────────────────────────────────────────────────────
df_unique = spark.createDataFrame([(1,), (2,), (3,)], ["id"])
df_dups   = spark.createDataFrame([(1,), (1,), (3,)], ["id"])

_assert_pass("unique: no duplicates passes",
    _run({"id":"r","description":"x","type":"unique","columns":["id"],"severity":"CRITICAL"}, df_unique))
_assert_fail("unique: duplicates fails",
    _run({"id":"r","description":"x","type":"unique","columns":["id"],"severity":"CRITICAL"}, df_dups))

# ── row_count ─────────────────────────────────────────────────────────────
df_three = spark.createDataFrame([(1,), (2,), (3,)], ["id"])

_assert_pass("row_count: within bounds passes",
    _run({"id":"r","description":"x","type":"row_count","min":1,"max":10,"severity":"CRITICAL"}, df_three))
_assert_fail("row_count: below min fails",
    _run({"id":"r","description":"x","type":"row_count","min":10,"severity":"CRITICAL"}, df_three))
_assert_fail("row_count: above max fails",
    _run({"id":"r","description":"x","type":"row_count","max":2,"severity":"WARNING"}, df_three))

# ── accepted_values ──────────────────────────────────────────────────────
df_av_ok  = spark.createDataFrame([("ACTIVE",), ("INACTIVE",)], ["status"])
df_av_bad = spark.createDataFrame([("ACTIVE",), ("DELETED",)],  ["status"])

_assert_pass("accepted_values: all valid passes",
    _run({"id":"r","description":"x","type":"accepted_values","column":"status","values":["ACTIVE","INACTIVE"],"severity":"WARNING"}, df_av_ok))
_assert_fail("accepted_values: invalid value fails",
    _run({"id":"r","description":"x","type":"accepted_values","column":"status","values":["ACTIVE","INACTIVE"],"severity":"WARNING"}, df_av_bad))

# ── regex ─────────────────────────────────────────────────────────────────
df_email_ok  = spark.createDataFrame([("a@b.com",), ("x@y.org",)], ["email"])
df_email_bad = spark.createDataFrame([("a@b.com",), ("not-an-email",)], ["email"])

_assert_pass("regex: valid pattern passes",
    _run({"id":"r","description":"x","type":"regex","column":"email","pattern":r"^[^@]+@[^@]+\.[^@]+$","severity":"WARNING"}, df_email_ok))
_assert_fail("regex: invalid value fails",
    _run({"id":"r","description":"x","type":"regex","column":"email","pattern":r"^[^@]+@[^@]+\.[^@]+$","severity":"WARNING"}, df_email_bad))

# ── range ─────────────────────────────────────────────────────────────────
df_range_ok  = spark.createDataFrame([(50.0,), (99.9,)], ["amount"])
df_range_bad = spark.createDataFrame([(50.0,), (200.0,)], ["amount"])

_assert_pass("range: within bounds passes",
    _run({"id":"r","description":"x","type":"range","column":"amount","min":0,"max":100,"severity":"WARNING"}, df_range_ok))
_assert_fail("range: above max fails",
    _run({"id":"r","description":"x","type":"range","column":"amount","min":0,"max":100,"severity":"WARNING"}, df_range_bad))

# ── completeness ──────────────────────────────────────────────────────────
df_complete  = spark.createDataFrame([(1, "A"), (2, "B"), (3, "C")], ["id", "name"])
df_sparse    = spark.createDataFrame([(1, None), (2, None), (3, "C")], ["id", "name"])

_assert_pass("completeness: below threshold passes",
    _run({"id":"r","description":"x","type":"completeness","column":"name","max_null_rate":0.1,"severity":"WARNING"}, df_complete))
_assert_fail("completeness: above threshold fails",
    _run({"id":"r","description":"x","type":"completeness","column":"name","max_null_rate":0.1,"severity":"WARNING"}, df_sparse))

# ── duplicate_count ───────────────────────────────────────────────────────
df_no_dups  = spark.createDataFrame([(1,), (2,), (3,)], ["id"])
df_has_dups = spark.createDataFrame([(1,), (1,), (2,)], ["id"])

_assert_pass("duplicate_count: 0 dups passes (max=0)",
    _run({"id":"r","description":"x","type":"duplicate_count","columns":["id"],"max_duplicates":0,"severity":"CRITICAL"}, df_no_dups))
_assert_fail("duplicate_count: dups exceed threshold fails",
    _run({"id":"r","description":"x","type":"duplicate_count","columns":["id"],"max_duplicates":0,"severity":"CRITICAL"}, df_has_dups))

# ── custom_sql ────────────────────────────────────────────────────────────
df_any = spark.createDataFrame([(1, 100.0), (2, 50.0)], ["id", "amount"])

_assert_pass("custom_sql: SQL returning pass=true passes",
    _run({
        "id":"r","description":"x","type":"custom_sql","severity":"WARNING",
        "sql": "SELECT COUNT(*) = 0 AS pass, COUNT(*) AS row_count FROM {table} WHERE amount < 0"
    }, df_any))
_assert_fail("custom_sql: SQL returning pass=false fails",
    _run({
        "id":"r","description":"x","type":"custom_sql","severity":"WARNING",
        "sql": "SELECT COUNT(*) = 0 AS pass, COUNT(*) AS row_count FROM {table} WHERE amount > 0"
    }, df_any))

In [0]:
# ── NEW RULE TYPES ─────────────────────────────────────────────────────────

# ── conditional_not_null ──────────────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, StringType as ST, DoubleType

df_cnn_ok = spark.createDataFrame(
    [("PROMO", 10.0), ("STANDARD", None), ("PROMO", 5.0)],
    ["order_type", "discount"]
)
df_cnn_bad = spark.createDataFrame(
    [("PROMO", None), ("STANDARD", None)],   # PROMO row missing discount
    ["order_type", "discount"]
)
cnn_rule = {
    "id": "r", "description": "x",
    "type": "conditional_not_null",
    "column": "discount",
    "condition_column": "order_type",
    "condition_value": "PROMO",
    "condition_operator": "eq",
    "severity": "WARNING",
}
_assert_pass("conditional_not_null: condition rows have values → passes", _run(cnn_rule, df_cnn_ok))
_assert_fail("conditional_not_null: condition rows have nulls → fails",   _run(cnn_rule, df_cnn_bad))

# Test with is_not_null operator (no condition_value needed)
df_cnn_flag_ok  = spark.createDataFrame([(1, "X", "FILLED"), (2, None, None)], ["id", "flag", "notes"])
df_cnn_flag_bad = spark.createDataFrame([(1, "X", None)],                      ["id", "flag", "notes"])
cnn_is_not_null_rule = {
    "id":"r","description":"x",
    "type":"conditional_not_null",
    "column":"notes",
    "condition_column":"flag",
    "condition_operator":"is_not_null",
    "severity":"WARNING",
}
_assert_pass("conditional_not_null: is_not_null operator passes",  _run(cnn_is_not_null_rule, df_cnn_flag_ok))
_assert_fail("conditional_not_null: is_not_null operator fails",   _run(cnn_is_not_null_rule, df_cnn_flag_bad))

# Test with 'in' operator
df_cnn_in_ok  = spark.createDataFrame([("VIP", 10.0), ("STANDARD", None)], ["tier", "bonus"])
df_cnn_in_bad = spark.createDataFrame([("VIP", None), ("GOLD", None)],      ["tier", "bonus"])
cnn_in_rule = {
    "id":"r","description":"x",
    "type":"conditional_not_null",
    "column":"bonus",
    "condition_column":"tier",
    "condition_value":["VIP","GOLD"],
    "condition_operator":"in",
    "severity":"WARNING",
}
_assert_pass("conditional_not_null: in operator passes",  _run(cnn_in_rule, df_cnn_in_ok))
_assert_fail("conditional_not_null: in operator fails",   _run(cnn_in_rule, df_cnn_in_bad))

# ── mutual_exclusivity ────────────────────────────────────────────────────
df_me_ok  = spark.createDataFrame(
    [("CC123", None, None), (None, "BA456", None)],
    ["credit_card_id", "bank_account_id", "wallet_id"]
)
df_me_bad_multi = spark.createDataFrame(
    [("CC123", "BA456", None)],   # two columns populated — violation
    ["credit_card_id", "bank_account_id", "wallet_id"]
)
df_me_bad_none = spark.createDataFrame(
    [(None, None, None)],         # no column populated — violation when allow_all_null=false
    ["credit_card_id", "bank_account_id", "wallet_id"]
)
me_rule = {
    "id":"r","description":"x",
    "type":"mutual_exclusivity",
    "columns":["credit_card_id","bank_account_id","wallet_id"],
    "allow_all_null": False,
    "severity":"CRITICAL",
}
_assert_pass("mutual_exclusivity: exactly one populated → passes", _run(me_rule, df_me_ok))
_assert_fail("mutual_exclusivity: two populated → fails",          _run(me_rule, df_me_bad_multi))
_assert_fail("mutual_exclusivity: all null → fails (allow_all_null=false)", _run(me_rule, df_me_bad_none))

# allow_all_null=true
me_rule_allow = {**me_rule, "allow_all_null": True}
_assert_pass("mutual_exclusivity: all null passes when allow_all_null=true",
    _run(me_rule_allow, df_me_bad_none))

# ── character_set ─────────────────────────────────────────────────────────
df_cs_ok  = spark.createDataFrame([("ABC123",), ("XYZ789",)], ["code"])
df_cs_bad = spark.createDataFrame([("ABC123",), ("XYZ 789!",)], ["code"])  # space + !
cs_rule_named = {
    "id":"r","description":"x",
    "type":"character_set",
    "column":"code",
    "charset_name":"alphanumeric",
    "severity":"WARNING",
}
_assert_pass("character_set (alphanumeric): all chars valid → passes", _run(cs_rule_named, df_cs_ok))
_assert_fail("character_set (alphanumeric): space+! found → fails",   _run(cs_rule_named, df_cs_bad))

# Explicit allowed_chars mode
cs_rule_explicit = {
    "id":"r","description":"x",
    "type":"character_set",
    "column":"code",
    "allowed_chars":"ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789",
    "severity":"WARNING",
}
_assert_pass("character_set (explicit): uppercase+digits → passes", _run(cs_rule_explicit, df_cs_ok))
_assert_fail("character_set (explicit): lowercase fails",
    _run(cs_rule_explicit, spark.createDataFrame([("abc",)], ["code"])))

# ── length_check ──────────────────────────────────────────────────────────
df_len_ok  = spark.createDataFrame([("12345678",), ("ABCDEFGHIJ",)], ["phone"])  # 8 and 10 chars
df_len_bad = spark.createDataFrame([("123",), ("12345678901234567890",)], ["phone"])  # 3 and 20 chars
len_rule = {
    "id":"r","description":"x",
    "type":"length_check",
    "column":"phone",
    "min_length": 7,
    "max_length": 15,
    "severity":"WARNING",
}
_assert_pass("length_check: within bounds → passes", _run(len_rule, df_len_ok))
_assert_fail("length_check: outside bounds → fails", _run(len_rule, df_len_bad))

# Exact length (2-char country code)
df_cc_ok  = spark.createDataFrame([("US",), ("GB",)], ["country_code"])
df_cc_bad = spark.createDataFrame([("USA",), ("GB",)], ["country_code"])
_assert_pass("length_check: exact 2-char → passes",
    _run({"id":"r","description":"x","type":"length_check","column":"country_code","min_length":2,"max_length":2,"severity":"CRITICAL"}, df_cc_ok))
_assert_fail("length_check: 3-char fails exact-2 rule",
    _run({"id":"r","description":"x","type":"length_check","column":"country_code","min_length":2,"max_length":2,"severity":"CRITICAL"}, df_cc_bad))

# strip_before_check
df_padded = spark.createDataFrame([("  AB  ",)], ["code"])   # 6 chars with spaces; stripped = 2 chars
_assert_pass("length_check: strip_before_check strips whitespace before measuring",
    _run({"id":"r","description":"x","type":"length_check","column":"code","min_length":1,"max_length":4,"strip_before_check":True,"severity":"INFO"}, df_padded))

# bytes mode
df_bytes = spark.createDataFrame([("hello",)], ["txt"])   # 5 bytes in ASCII
_assert_pass("length_check: bytes mode within bounds",
    _run({"id":"r","description":"x","type":"length_check","column":"txt","min_length":3,"max_length":10,"count_mode":"bytes","severity":"INFO"}, df_bytes))

## 3. Final Summary

In [0]:
sep = "=" * 60
print(f"\n{sep}")
print(f"  TEST RESULTS")
print(sep)
print(f"  ✅ Passed : {_passed}")
print(f"  ❌ Failed : {_failed}")
print(f"  Total    : {_passed + _failed}")
print(sep)

if _failed > 0:
    raise Exception(f"TEST SUITE FAILED: {_failed} test(s) failed. See output above.")
else:
    print("\n  🎉 All tests passed!")